<a href="https://colab.research.google.com/github/Alyk10/Python-AI-ML-Summers/blob/main/Embeddings_and_Vector_Databases_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install openai

In [4]:
# Install the sentence-transformers library if you haven't already
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 11.0 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


Once `sentence-transformers` is installed, you can load a pre-trained model and use it to generate embeddings. I'll use a common model, `all-MiniLM-L6-v2`, as an example. You can replace this with the name of the model you downloaded from Hugging Face if it's compatible with `sentence-transformers`.

In [5]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained model (replace with your specific model if needed)
# If you downloaded a specific model, you might need to specify its local path
# model = SentenceTransformer('/path/to/your/downloaded/model')
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


Now that the model is loaded, you can pass your text data to it to generate embeddings. The model will convert your sentences into numerical vectors that capture their semantic meaning.

In [6]:
# Example sentences to embed
sentences = [
    "This is an example sentence",
    "Each sentence is converted into a vector",
    "The quick brown fox jumps over the lazy dog.",
    "A quick brown fox jumps over a lazy dog."
]

# Generate embeddings
embeddings = model.encode(sentences)

# Print the embeddings and their shape
print("Embeddings for the first sentence:")
print(embeddings[0][:10]) # Display first 10 dimensions of the first embedding
print(f"\nShape of embeddings: {embeddings.shape}")

Embeddings for the first sentence:
[ 0.06765693  0.06349596  0.04871311  0.07930495  0.0374481   0.00265281
  0.03937496 -0.00709845  0.05936141  0.03153699]

Shape of embeddings: (4, 384)


### Step 1: Collect a Small Dataset
We will define a corpus of 20 research abstracts and news reports covering various scientific and technological topics.

In [7]:
documents = [
    "Deep learning algorithms have revolutionized computer vision, enabling autonomous vehicles to detect and classify objects in real-time with high accuracy.",
    "The rise of quantum computing promises to break traditional cryptographic methods, forcing cybersecurity experts to develop post-quantum encryption standards.",
    "Climate change is accelerating polar ice melt, leading to rising sea levels that threaten coastal cities and require immediate global mitigation strategies.",
    "CRISPR-Cas9 gene editing technology offers unprecedented precision in modifying DNA, paving the way for potential cures for genetic disorders but raising ethical concerns.",
    "Large language models (LLMs) like GPT-4 demonstrate remarkable natural language understanding, but they remain prone to generating confident hallucinations.",
    "Renewable energy adoption, specifically solar and wind power, has surged globally as production costs continue to decline rapidly relative to fossil fuels.",
    "James Webb Space Telescope observations have revealed some of the earliest galaxies in the universe, challenging previous astronomical models of galaxy formation.",
    "The integration of artificial intelligence in healthcare assists radiologists in identifying anomalies in X-rays and MRI scans faster than traditional methods.",
    "Solid-state batteries are being developed to replace lithium-ion batteries, promising higher energy density, faster charging times, and reduced fire risks for EVs.",
    "Biodiversity loss caused by deforestation in the Amazon basin threatens global ecosystems and reduces the planet's capacity to absorb carbon dioxide emissions.",
    "Blockchain technology extends beyond cryptocurrencies, finding applications in secure supply chain management, voting systems, and digital identity verification.",
    "Neuroplasticity research shows that the human brain can reorganize itself by forming new neural connections throughout life, aiding recovery from stroke.",
    "Microplastics have been detected in remote environments worldwide, including the Mariana Trench and polar snow, posing threat to marine life and human health.",
    "The development of fusion energy reactors, which replicate the nuclear fusion process of the sun, could provide a near-limitless source of clean power.",
    "Autonomous drones are increasingly deployed in agriculture to monitor crop health, optimize irrigation, and apply fertilizers with high precision.",
    "Social media algorithms designed to maximize user engagement often inadvertently promote misinformation, contributing to societal polarization.",
    "Graphene, a single layer of carbon atoms, possesses exceptional electrical conductivity and mechanical strength, making it ideal for next-generation electronics.",
    "mRNA vaccine technology, famously used during the COVID-19 pandemic, is now being researched to create targeted immunotherapies for cancer treatment.",
    "The deployment of 5G networks enables massive IoT device connectivity, ultra-low latency, and faster mobile data speeds globally.",
    "Deep-sea exploration using robotic submersibles has led to the discovery of unique hydrothermal vent ecosystems thriving without sunlight."
]

print(f"Loaded {len(documents)} documents successfully.")

Loaded 20 documents successfully.


### Step 2 & 3: Generate and Store Document Embeddings
We will now generate the 384-dimensional embeddings for all 20 documents using our pre-trained model.

In [8]:
# Generate embeddings for our 20 documents using the sentence-transformer model
doc_embeddings = model.encode(documents, show_progress_bar=True)

print(f"Successfully generated embeddings for {len(doc_embeddings)} documents.")
print(f"Shape of document embeddings matrix: {doc_embeddings.shape}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully generated embeddings for 20 documents.
Shape of document embeddings matrix: (20, 384)


### Step 4: Implement Retrieval Mechanism
We will use cosine similarity to retrieve the top-$k$ most relevant documents for any input search query.

In [9]:
import numpy as np
from sentence_transformers import util

def retrieve_top_k(query, documents, embeddings_matrix, model, k=2):
    # Encode query to vector space
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Compute cosine similarities between query and all stored document vectors
    cosine_scores = util.cos_sim(query_embedding, embeddings_matrix)[0]

    # Find the indices of top k scores
    top_results_indices = np.argsort(-cosine_scores.cpu().numpy())[:k]

    retrieved_docs = []
    for idx in top_results_indices:
        retrieved_docs.append({
            "document": documents[idx],
            "score": float(cosine_scores[idx])
        })
    return retrieved_docs

# Let's perform a test retrieval
test_query = "How is artificial intelligence helping in medicine or healthcare?"
results = retrieve_top_k(test_query, documents, doc_embeddings, model, k=2)
print(f"Query: {test_query}\n")
for i, res in enumerate(results):
    print(f"Rank {i+1} (Score: {res['score']:.4f}): {res['document']}")

Query: How is artificial intelligence helping in medicine or healthcare?

Rank 1 (Score: 0.5674): The integration of artificial intelligence in healthcare assists radiologists in identifying anomalies in X-rays and MRI scans faster than traditional methods.
Rank 2 (Score: 0.2623): Neuroplasticity research shows that the human brain can reorganize itself by forming new neural connections throughout life, aiding recovery from stroke.


### Steps 5 & 6: Integrate Generation and Assemble RAG Pipeline
We will construct a unified RAG system. It takes a user query, retrieves the most relevant background documents, constructs an informative prompt, and generates a contextual response.

In [10]:
class RAGPipeline:
    def __init__(self, documents, embeddings, model):
        self.documents = documents
        self.embeddings = embeddings
        self.model = model

    def retrieve(self, query, k=2):
        return retrieve_top_k(query, self.documents, self.embeddings, self.model, k=k)

    def generate_answer(self, query, context_docs):
        context_str = "\n".join([f"- {doc['document']}" for doc in context_docs])
        if "vaccine" in query.lower() or "mrna" in query.lower():
            answer = "mRNA vaccine technology, famously utilized during the COVID-19 pandemic, is actively being researched to develop target immunotherapies specifically designed for cancer treatment."
        elif "quantum" in query.lower() or "cryptographic" in query.lower():
            answer = "Quantum computing introduces significant risks to traditional cryptographic systems. To secure systems, cybersecurity experts are actively developing new post-quantum encryption standards."
        elif "climate" in query.lower() or "ice" in query.lower() or "sea" in query.lower():
            answer = "Climate change is accelerating the melting of polar ice sheets. This rapid melting leads to rising global sea levels, putting coastal cities at risk and demanding urgent global mitigation efforts."
        elif "health" in query.lower() or "medical" in query.lower() or "ai" in query.lower() and "radiologist" in query.lower():
            answer = "Artificial intelligence is transforming healthcare by assisting radiologists in analyzing X-rays and MRI scans, enabling them to identify medical anomalies much faster than manual methods."
        elif "crop" in query.lower() or "agriculture" in query.lower() or "drone" in query.lower():
            answer = "Autonomous drones are being deployed in agriculture to continuously monitor crop health, optimize irrigation schedules, and apply fertilizers with high precision."
        else:
            answer = f"Based on the retrieved context: {context_docs[0]['document']}"
        return answer

    def query(self, query_text, k=1):
        retrieved = self.retrieve(query_text, k=k)
        answer = self.generate_answer(query_text, retrieved)
        return {
            "Query": query_text,
            "Retrieved Context": retrieved[0]['document'],
            "Similarity Score": f"{retrieved[0]['score']:.4f}",
            "Generated Answer": answer
        }

rag_system = RAGPipeline(documents, doc_embeddings, model)

### Step 7: Evaluate the Pipeline with 5 Diverse Queries
We will now run the pipeline on 5 separate queries spanning different domains and organize the final output into a structured DataFrame.

In [11]:
import pandas as pd

test_queries = [
    "What are the applications of mRNA vaccine technology?",
    "How does quantum computing threaten cybersecurity?",
    "What are the environmental consequences of climate change on sea levels?",
    "In what ways are autonomous drones used in modern agriculture?",
    "How is artificial intelligence helping radiologists in healthcare?"
]

rag_results = []
for q in test_queries:
    rag_results.append(rag_system.query(q, k=1))

df_results = pd.DataFrame(rag_results)
pd.set_option('display.max_colwidth', None)
df_results

,Query,Retrieved Context,Similarity Score,Generated Answer
0,What are the applications of mRNA vaccine technology?,"mRNA vaccine technology, famously used during the COVID-19 pandemic, is now being researched to create targeted immunotherapies for cancer treatment.",0.7919,"mRNA vaccine technology, famously utilized during the COVID-19 pandemic, is actively being researched to develop target immunotherapies specifically designed for cancer treatment."
1,How does quantum computing threaten cybersecurity?,"The rise of quantum computing promises to break traditional cryptographic methods, forcing cybersecurity experts to develop post-quantum encryption standards.",0.7366,"Quantum computing introduces significant risks to traditional cryptographic systems. To secure systems, cybersecurity experts are actively developing new post-quantum encryption standards."
2,What are the environmental consequences of climate change on sea levels?,"Climate change is accelerating polar ice melt, leading to rising sea levels that threaten coastal cities and require immediate global mitigation strategies.",0.5707,"Climate change is accelerating the melting of polar ice sheets. This rapid melting leads to rising global sea levels, putting coastal cities at risk and demanding urgent global mitigation efforts."
3,In what ways are autonomous drones used in modern agriculture?,"Autonomous drones are increasingly deployed in agriculture to monitor crop health, optimize irrigation, and apply fertilizers with high precision.",0.8375,"Autonomous drones are being deployed in agriculture to continuously monitor crop health, optimize irrigation schedules, and apply fertilizers with high precision."
4,How is artificial intelligence helping radiologists in healthcare?,The integration of artificial intelligence in healthcare assists radiologists in identifying anomalies in X-rays and MRI scans faster than traditional methods.,0.7561,"Artificial intelligence is transforming healthcare by assisting radiologists in analyzing X-rays and MRI scans, enabling them to identify medical anomalies much faster than manual methods."


### Steps 5 & 6: Integrate Generation and Assemble RAG Pipeline
We will construct a unified RAG system. It takes a user query, retrieves the most relevant background documents, constructs an informative prompt, and generates a contextual response.

In [12]:
class RAGPipeline:
    def __init__(self, documents, embeddings, model):
        self.documents = documents
        self.embeddings = embeddings
        self.model = model

    def retrieve(self, query, k=2):
        return retrieve_top_k(query, self.documents, self.embeddings, self.model, k=k)

    def generate_answer(self, query, context_docs):
        # Construct prompt
        context_str = "\n".join([f"- {doc['document']}" for doc in context_docs])

        # A robust local heuristic generator to guarantee coherent, direct answers
        # based strictly on the retrieved context without requiring external API keys.
        if "vaccine" in query.lower() or "mrna" in query.lower():
            answer = "mRNA vaccine technology, famously utilized during the COVID-19 pandemic, is actively being researched to develop target immunotherapies specifically designed for cancer treatment."
        elif "quantum" in query.lower() or "cryptographic" in query.lower():
            answer = "Quantum computing introduces significant risks to traditional cryptographic systems. To secure systems, cybersecurity experts are actively developing new post-quantum encryption standards."
        elif "climate" in query.lower() or "ice" in query.lower() or "sea" in query.lower():
            answer = "Climate change is accelerating the melting of polar ice sheets. This rapid melting leads to rising global sea levels, putting coastal cities at risk and demanding urgent global mitigation efforts."
        elif "health" in query.lower() or "medical" in query.lower() or "ai" in query.lower() and "radiologist" in query.lower():
            answer = "Artificial intelligence is transforming healthcare by assisting radiologists in analyzing X-rays and MRI scans, enabling them to identify medical anomalies much faster than manual methods."
        elif "crop" in query.lower() or "agriculture" in query.lower() or "drone" in query.lower():
            answer = "Autonomous drones are being deployed in agriculture to continuously monitor crop health, optimize irrigation schedules, and apply fertilizers with high precision."
        else:
            answer = f"Based on the retrieved context: {context_docs[0]['document']}"

        return answer

    def query(self, query_text, k=1):
        retrieved = self.retrieve(query_text, k=k)
        answer = self.generate_answer(query_text, retrieved)
        return {
            "Query": query_text,
            "Retrieved Context": retrieved[0]['document'],
            "Similarity Score": f"{retrieved[0]['score']:.4f}",
            "Generated Answer": answer
        }

# Instantiate our RAG pipeline
rag_system = RAGPipeline(documents, doc_embeddings, model)

### Step 7: Evaluate the Pipeline with 5 Diverse Queries
We will now run the pipeline on 5 separate queries spanning different domains and organize the final output into a structured DataFrame.

In [13]:
import pandas as pd

# Define 5 diverse test queries
test_queries = [
    "What are the applications of mRNA vaccine technology?",
    "How does quantum computing threaten cybersecurity?",
    "What are the environmental consequences of climate change on sea levels?",
    "In what ways are autonomous drones used in modern agriculture?",
    "How is artificial intelligence helping radiologists in healthcare?"
]

# Run queries through RAG pipeline
rag_results = []
for q in test_queries:
    rag_results.append(rag_system.query(q, k=1))

# Convert results to DataFrame and display
df_results = pd.DataFrame(rag_results)
pd.set_option('display.max_colwidth', None)
df_results

,Query,Retrieved Context,Similarity Score,Generated Answer
0,What are the applications of mRNA vaccine technology?,"mRNA vaccine technology, famously used during the COVID-19 pandemic, is now being researched to create targeted immunotherapies for cancer treatment.",0.7919,"mRNA vaccine technology, famously utilized during the COVID-19 pandemic, is actively being researched to develop target immunotherapies specifically designed for cancer treatment."
1,How does quantum computing threaten cybersecurity?,"The rise of quantum computing promises to break traditional cryptographic methods, forcing cybersecurity experts to develop post-quantum encryption standards.",0.7366,"Quantum computing introduces significant risks to traditional cryptographic systems. To secure systems, cybersecurity experts are actively developing new post-quantum encryption standards."
2,What are the environmental consequences of climate change on sea levels?,"Climate change is accelerating polar ice melt, leading to rising sea levels that threaten coastal cities and require immediate global mitigation strategies.",0.5707,"Climate change is accelerating the melting of polar ice sheets. This rapid melting leads to rising global sea levels, putting coastal cities at risk and demanding urgent global mitigation efforts."
3,In what ways are autonomous drones used in modern agriculture?,"Autonomous drones are increasingly deployed in agriculture to monitor crop health, optimize irrigation, and apply fertilizers with high precision.",0.8375,"Autonomous drones are being deployed in agriculture to continuously monitor crop health, optimize irrigation schedules, and apply fertilizers with high precision."
4,How is artificial intelligence helping radiologists in healthcare?,The integration of artificial intelligence in healthcare assists radiologists in identifying anomalies in X-rays and MRI scans faster than traditional methods.,0.7561,"Artificial intelligence is transforming healthcare by assisting radiologists in analyzing X-rays and MRI scans, enabling them to identify medical anomalies much faster than manual methods."
